# 02 特征工程：验证"没有泄漏、没有错位"

目标：确认 `lag0 / lag96 / y` 三个字段的**时间对齐**是对的。
特征写错不会报错，只会让指标悄悄变好看——所以这里用断言把它钉死。

In [1]:
# 自包含引导：从当前目录向上找到含 src/config.py 的仓库根，再挂到 sys.path
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / "src" / "config.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p / "src"))
print("仓库根:", p)

import numpy as np
import pandas as pd
import data_io, features as F, split as S

d = F.build_features(data_io.for_market(data_io.load_raw(), "DA"), h=1)
print("形状:", d.shape)
d[["trade_date", "delivery_time", "price", "lag0", "lag1", "lag96", "y"]].head(3)

仓库根: /Users/xian/Desktop/electricity-price
[features] h=1 丢弃 193 行（lag96 热身 + 末尾 1 行），保留 36959
形状: (36959, 22)


,trade_date,delivery_time,price,lag0,lag1,lag96,y
0,2024-07-04,2024-07-04 00:00:00+08:00,254.08,254.08,265.86,239.72,288.15
1,2024-07-04,2024-07-04 00:15:00+08:00,288.15,288.15,254.08,262.38,278.60
2,2024-07-04,2024-07-04 00:30:00+08:00,278.60,278.60,288.15,265.81,270.34


In [2]:
# 断言 1：lag0 = 本行价格（t0 时刻已出清的那一格）
assert np.allclose(d["lag0"].to_numpy(), d["price"].to_numpy())

# 断言 2：lag96 = 96 行之前的价格（即昨日同一时段）
assert np.allclose(d["lag96"].to_numpy()[96:], d["price"].to_numpy()[:-96])

# 断言 3：y = 下一行价格（h=1）
assert np.allclose(d["y"].to_numpy()[:-1], d["price"].to_numpy()[1:])

# 断言 4：不能有缺失，且特征里不许出现当期价格的"未来版"
assert d["y"].notna().all() and d[F.FEATURES].notna().all().all()
print("四条对齐断言全部通过")

四条对齐断言全部通过


In [3]:
# 相关性直觉：lag0 应该极强，但不是 1（是 1 就说明泄漏）
d[["lag0", "lag1", "lag96", "roll_mean_96"]].corrwith(d["y"]).round(4)

lag0            0.9320
lag1            0.9071
lag96           0.7387
roll_mean_96    0.7053
dtype: float64

In [4]:
parts, spans = S.time_split(d)
for k, v in spans.items():
    print(f"{k:5s}: {str(v[0])[:10]} → {str(v[1])[:10]}  ({v[2]} 行)")

train: 2024-07-04 → 2025-02-18  (22080 行)
valid: 2025-02-20 → 2025-05-06  (7296 行)
test : 2025-05-08 → 2025-07-23  (7391 行)
